# 1. Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
)

# 2. ตั้งค่า

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/SleepStageClassificationProject")
DATA_DIR = PROJECT_DIR / "dataset" / "preprocess_microvolt_multi5"
RES = PROJECT_DIR / "results"

SEEDS = [42, 43, 44]
EXP23_DIRS = {s: RES / f"CNN_5_7ch_Exp23_seed{s}" for s in SEEDS}
EXP13_DIRS = {
    42: RES / "CNN_5_7ch_Exp13_seed42",
    43: RES / "CNN_5_7ch_Exp13_Seed43",
    44: RES / "CNN_5_7ch_Exp13_Seed44",
}

ANALYSIS_DIR = RES / "CNN_5_7ch_Exp23_analysis" / "subject_error_analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

STAGES = ["W", "N1", "N2", "N3", "R"]
LABELS = [0, 1, 2, 3, 4]
CHANNELS = ["F4-M1", "C4-M1", "O2-M1", "C3-M2", "E1-M2", "E2-M2", "Chin"]
TARGET_SUBJECTS = ["SN125", "SN152"]

KEY_COLS = ["subject", "epoch_start", "epoch_end", "actual"]
PROB_COLS = ["prob_" + s for s in STAGES]

# 3. โหลด val_predictions ของ Exp23 และสร้าง Soft Ensemble

In [ ]:
def load_preds(dirs):
    d = {s: pd.read_csv(dirs[s] / "val_predictions.csv") for s in SEEDS}
    base = d[SEEDS[0]]
    for s, df in d.items():
        assert len(df) == len(base), f"seed {s}: จำนวนแถวไม่ตรง"
        for c in KEY_COLS:
            assert (df[c].astype(str).to_numpy() == base[c].astype(str).to_numpy()).all(), \
                f"seed {s}: คอลัมน์ {c} ไม่ตรงกัน"
    return d

def make_soft(d):
    base = d[SEEDS[0]]
    soft = base[KEY_COLS].copy()
    P = np.mean([d[s][PROB_COLS].to_numpy() for s in SEEDS], axis=0)
    soft[PROB_COLS] = P
    soft["predicted"] = np.array(STAGES)[P.argmax(axis=1)]
    return soft

d23 = load_preds(EXP23_DIRS)
soft23 = make_soft(d23)

MODELS = {f"Seed {s}": d23[s] for s in SEEDS}
MODELS["Soft Ensemble"] = soft23
pred_df = soft23

chk = soft23
print("Soft check -> Acc %.4f | Macro-F1 %.4f | Kappa %.4f" % (
    accuracy_score(chk.actual, chk.predicted),
    f1_score(chk.actual, chk.predicted, labels=STAGES, average="macro"),
    cohen_kappa_score(chk.actual, chk.predicted)))
print("Sequences:", len(pred_df), "| Subjects:", pred_df["subject"].nunique())

# 4. โหลด X_val และตรวจการเรียงตรงกับ val_predictions

In [ ]:
x_path = None
for p in [EXP23_DIRS[s] / "X_val.npy" for s in SEEDS] + \
         [EXP13_DIRS[s] / "X_val.npy" for s in SEEDS] + \
         [RES / "CNN_5_7ch_Exp13_PATIENCE20_seed42" / "X_val.npy"]:
    if p.exists():
        x_path = p
        break
assert x_path is not None, "ไม่พบ X_val.npy ในโฟลเดอร์ใดเลย"
X_val = np.load(x_path, mmap_mode="r")
print("X_val จาก:", x_path, "| shape:", X_val.shape)

meta_rows = []
for path in sorted((DATA_DIR / "val").glob("*.npz")):
    subject = path.stem
    with np.load(path, allow_pickle=True) as data:
        for g, stage in enumerate(data["stage"]):
            stage = str(stage)
            if stage not in STAGES or int(data["n_valid"][g]) != 5:
                continue
            y = np.asarray(data["y"][g]).astype(str).ravel()
            ep = np.asarray(data["epoch_indices"][g]).astype(int).ravel()
            if len(y) != 5 or not np.all(y == stage) or len(ep) != 5 or not np.all(np.diff(ep) == 1):
                continue
            meta_rows.append({"subject": subject, "stage": stage,
                              "epoch_start": int(ep[0]), "epoch_end": int(ep[-1])})
meta_val = pd.DataFrame(meta_rows)

assert len(X_val) == len(pred_df) == len(meta_val)
assert np.array_equal(meta_val["subject"], pred_df["subject"])
assert np.array_equal(meta_val["stage"], pred_df["actual"])
assert np.array_equal(meta_val["epoch_start"], pred_df["epoch_start"])
assert np.array_equal(meta_val["epoch_end"], pred_df["epoch_end"])
print("ตรวจการเรียงข้อมูลผ่าน")

# 5. Accuracy / Kappa รายคน ของทุกโมเดล

In [ ]:
def subj_metrics(df, name):
    rows = []
    for subj, g in df.groupby("subject"):
        rows.append({"model": name, "subject": subj, "n_sequences": len(g),
                     "accuracy": accuracy_score(g.actual, g.predicted),
                     "kappa": cohen_kappa_score(g.actual, g.predicted),
                     "n_errors": int((g.actual != g.predicted).sum())})
    return pd.DataFrame(rows)

subject_long = pd.concat([subj_metrics(df, n) for n, df in MODELS.items()], ignore_index=True)
acc_wide = subject_long.pivot(index="subject", columns="model", values="accuracy")
acc_wide["n_sequences"] = subject_long[subject_long.model == "Soft Ensemble"].set_index("subject")["n_sequences"]
acc_wide = acc_wide.sort_values("Soft Ensemble")
print("Accuracy รายคน (เรียงจากต่ำสุดของ Soft Ensemble) -- ครบ 21 คน")
display(acc_wide.round(4))
kappa_wide = subject_long.pivot(index="subject", columns="model", values="kappa").loc[acc_wide.index]
print("Kappa รายคน (ไม่เสถียรเมื่อคนนั้นมีบางคลาสน้อย ให้อ่าน Accuracy เป็นหลัก)")
display(kappa_wide.round(4))

subject_long.to_csv(ANALYSIS_DIR / "subject_metrics_long.csv", index=False)
acc_wide.to_csv(ANALYSIS_DIR / "subject_accuracy_wide.csv")
kappa_wide.to_csv(ANALYSIS_DIR / "subject_kappa_wide.csv")

# 6. ข้อผิดพลาดกระจุกที่ subject ไหน (ตัดสิน P1 vs P2)

In [ ]:
a, p = pred_df["actual"], pred_df["predicted"]
CATS = {
    "N2->N3": (a == "N2") & (p == "N3"),
    "N3->N2": (a == "N3") & (p == "N2"),
    "R->N1":  (a == "R") & (p == "N1"),
    "R->N2":  (a == "R") & (p == "N2"),
    "R->N3":  (a == "R") & (p == "N3"),
    "N2->N1": (a == "N2") & (p == "N1"),
    "W->N1":  (a == "W") & (p == "N1"),
    "N1->other": (a == "N1") & (p != "N1"),
}
err = pd.DataFrame({k: v.groupby(pred_df["subject"]).sum() for k, v in CATS.items()})
err["total_errors"] = (a != p).groupby(pred_df["subject"]).sum()
err["n_sequences"] = pred_df.groupby("subject").size()
err["error_rate"] = err["total_errors"] / err["n_sequences"]
err["share_of_all_errors"] = err["total_errors"] / err["total_errors"].sum()
err["share_of_data"] = err["n_sequences"] / err["n_sequences"].sum()
err = err.sort_values("total_errors", ascending=False)
print("ข้อผิดพลาดรายคนของ Soft Ensemble (เรียงจากผิดมากสุด)")
display(err.round(3))
err.to_csv(ANALYSIS_DIR / "error_by_subject_soft.csv")

summ = []
for k in [2, 3, 5]:
    top = err.head(k)
    summ.append({"top_k_subjects": k,
                 "share_of_errors": top["share_of_all_errors"].sum(),
                 "share_of_data": top["share_of_data"].sum()})
print("สัดส่วนข้อผิดพลาดของ top-k subjects เทียบสัดส่วนข้อมูล")
display(pd.DataFrame(summ).round(3))

cat_cols = list(CATS.keys())
cat_share = pd.DataFrame({
    c: [err[c].sort_values(ascending=False).head(3).sum() / max(err[c].sum(), 1)]
    for c in cat_cols}, index=["share ของ top-3 subjects"]).T
cat_share["total"] = err[cat_cols].sum().astype(int)
print("แต่ละประเภทข้อผิดพลาด top-3 subjects ถือกี่ %")
display(cat_share.round(3))
pd.DataFrame(summ).to_csv(ANALYSIS_DIR / "error_concentration_summary.csv", index=False)
cat_share.to_csv(ANALYSIS_DIR / "error_category_concentration.csv")

# 9. ผลเมื่อตัด subject ออก (ทุกโมเดล, เฉพาะการวิเคราะห์ผลกระทบ)

In [ ]:
def overall(df):
    return {"subjects": df["subject"].nunique(), "sequences": len(df),
            "accuracy": accuracy_score(df.actual, df.predicted),
            "macro_f1": f1_score(df.actual, df.predicted, labels=STAGES, average="macro"),
            "kappa": cohen_kappa_score(df.actual, df.predicted)}

rows = []
for name, df in MODELS.items():
    rows.append({"model": name, "condition": "Full", **overall(df)})
    rows.append({"model": name, "condition": "Excl. SN125+SN152",
                 **overall(df[~df.subject.isin(TARGET_SUBJECTS)])})
excl = pd.DataFrame(rows)
print("ผลกระทบของการตัด SN125/SN152 (ไม่ใช่ผลประเมินหลัก)")
display(excl.round(4))
excl.to_csv(ANALYSIS_DIR / "exclusion_comparison_all_models.csv", index=False)

# 10. Confusion Matrix ของ SN125 / SN152 (Soft Ensemble)

In [ ]:
for subject in TARGET_SUBJECTS:
    g = pred_df[pred_df.subject == subject]
    cm = confusion_matrix(g.actual, g.predicted, labels=STAGES)
    cm_df = pd.DataFrame(cm, index=STAGES, columns=STAGES).rename_axis("Actual").rename_axis("Predicted", axis=1)
    print(subject, "| n =", len(g), "| Acc = %.4f" % accuracy_score(g.actual, g.predicted))
    display(cm_df)
    cm_df.to_csv(ANALYSIS_DIR / f"{subject}_confusion_matrix_soft.csv")

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(cm, display_labels=STAGES).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
    ax.set_title(f"{subject} - Exp23 Soft Ensemble")
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / f"{subject}_confusion_matrix_soft.png", dpi=300)
    plt.show()


# 11. (ไม่บังคับ) เทียบ Exp13 Soft กับ Exp23 Soft รายคน แบบจับคู่

In [ ]:
try:
    d13 = load_preds(EXP13_DIRS)
    soft13 = make_soft(d13)
    assert (soft13[KEY_COLS].astype(str).to_numpy() == soft23[KEY_COLS].astype(str).to_numpy()).all()
    s13 = subj_metrics(soft13, "Exp13 Soft").set_index("subject")
    s23 = subj_metrics(soft23, "Exp23 Soft").set_index("subject")
    paired = pd.DataFrame({
        "n_sequences": s23["n_sequences"],
        "acc_exp13": s13["accuracy"], "acc_exp23": s23["accuracy"],
        "delta_acc": s23["accuracy"] - s13["accuracy"],
        "errors_exp13": s13["n_errors"], "errors_exp23": s23["n_errors"],
    }).sort_values("delta_acc")
    print("Exp23 Soft - Exp13 Soft รายคน (delta_acc > 0 = Exp23 ดีกว่า)")
    display(paired.round(4))
    print("จำนวน subject ที่ Exp23 ดีกว่า/เท่า/แย่กว่า:",
          int((paired.delta_acc > 0).sum()), "/", int((paired.delta_acc == 0).sum()), "/",
          int((paired.delta_acc < 0).sum()))
    paired.to_csv(ANALYSIS_DIR / "paired_exp13_vs_exp23_soft_by_subject.csv")
except Exception as e:
    print("ข้ามส่วนเทียบ Exp13:", e)

print("บันทึกผลทั้งหมดไว้ที่:", ANALYSIS_DIR)

In [ ]:
for name, df in [("Exp13 Soft", soft13), ("Exp23 Soft", soft23)]:
    g = df[df.subject == "SN063"]
    cm = confusion_matrix(g.actual, g.predicted, labels=STAGES)
    print(name, "| n =", len(g), "| Acc = %.4f" % accuracy_score(g.actual, g.predicted))
    display(pd.DataFrame(cm, index=STAGES, columns=STAGES))